### Data Ingestion

In [99]:
### document data structure

from langchain_core.documents import Document

In [100]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Tejesh Verma",
        "date_created":"2025-01-01"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Tejesh Verma', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [101]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [102]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("sample text files created")

sample text files created


In [103]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document) 

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [104]:
### Directory loader
from langchain_community.document_loaders import DirectoryLoader

#load all the text files from directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## pattern to match files
    loader_cls= TextLoader, ## loader class to use
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True
)
text_document=dir_loader.load()
text_document

100%|██████████| 2/2 [00:00<00:00, 3466.37it/s]


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

In [105]:
### Directory loader
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader, PyPDFLoader

#load all the text files from directory
dir_loader = DirectoryLoader(
    "../data/pdfs",
    glob="**/*.pdf", ## pattern to match files
    loader_cls= PyMuPDFLoader, ## loader class to use
    show_progress=True
)
document=dir_loader.load()
document

100%|██████████| 1/1 [00:00<00:00, 10.03it/s]


[Document(metadata={'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with acmart 2023/11/05 v2.00 Typesetting articles for the Association for Computing Machinery and hyperref 2023-07-08 v7.01b Hypertext links for LaTeX', 'creationdate': '2024-04-05T22:57:47+00:00', 'source': '../data/pdfs/amazon-memorydb-a-fast-and-durable-memory-first-cloud-database.pdf', 'file_path': '../data/pdfs/amazon-memorydb-a-fast-and-durable-memory-first-cloud-database.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'Amazon MemoryDB: A Fast and Durable Memory-First Cloud Database', 'author': '', 'subject': '-  Information systems  ->  Main memory engines.', 'keywords': '', 'moddate': '2024-04-05T22:57:47+00:00', 'trapped': '', 'modDate': 'D:20240405225747Z', 'creationDate': 'D:20240405225747Z', 'page': 0}, page_content='Amazon MemoryDB: A Fast and Durable Memory-First Cloud\nDatabase\nYacine Taleb\nAmazon Web Services\nCanada\nKevin McGehee

In [106]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks = split_documents(document)

Split 12 documents into 88 chunks

Example chunk:
Content: Amazon MemoryDB: A Fast and Durable Memory-First Cloud
Database
Yacine Taleb
Amazon Web Services
Canada
Kevin McGehee
Amazon Web Services
USA
Nan Yan
Amazon Web Services
Canada
Shawn Wang
Amazon Web S...
Metadata: {'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with acmart 2023/11/05 v2.00 Typesetting articles for the Association for Computing Machinery and hyperref 2023-07-08 v7.01b Hypertext links for LaTeX', 'creationdate': '2024-04-05T22:57:47+00:00', 'source': '../data/pdfs/amazon-memorydb-a-fast-and-durable-memory-first-cloud-database.pdf', 'file_path': '../data/pdfs/amazon-memorydb-a-fast-and-durable-memory-first-cloud-database.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'Amazon MemoryDB: A Fast and Durable Memory-First Cloud Database', 'author': '', 'subject': '-  Information systems  ->  Main memory engines.', 'keywords': '', 'moddate': '2024-0

### embedding and vectorStoreDB

In [107]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [108]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        #Initialise the embedding manager
        # Args: model_name: HuggingFace model name for sentence embeddings
        
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        #load the sentenceTransformer model
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        # generate embeddings for a list of texts
        
        if not self.model:
            raise ValueError("Model Not Loaded")

        print(f"Generate embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
        

In [109]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9518.44it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [110]:
class VectorStore:
    def __init__(self, collection_name : str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialise_store()

    def _initialise_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client =  chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embedding for RAG"}
            )
            print(f"Vector store intialised, Collection {self.collection_name}")
            print(f"Existing document in collection {self.collection.count()}")
        except Exception as e:
            print(f"Error initialising vector store: {e}")
            raise

    def add_documents(self, documents: List[any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"adding {len(documents)} documents to vector store")

        #prepare data for ChromaDB
        ids = []
        metadatas = []
        document_text = []
        embedding_list = []

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #document content
            document_text.append(doc.page_content)

            #embedding
            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids = ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=document_text
            )

            print(f"Successfully added {len(documents)} documents to vector Store")
            print(f"Totla document in collection {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise




In [111]:
vectorStore = VectorStore()

Vector store intialised, Collection pdf_documents
Existing document in collection 34


In [112]:
chunks

[Document(metadata={'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with acmart 2023/11/05 v2.00 Typesetting articles for the Association for Computing Machinery and hyperref 2023-07-08 v7.01b Hypertext links for LaTeX', 'creationdate': '2024-04-05T22:57:47+00:00', 'source': '../data/pdfs/amazon-memorydb-a-fast-and-durable-memory-first-cloud-database.pdf', 'file_path': '../data/pdfs/amazon-memorydb-a-fast-and-durable-memory-first-cloud-database.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': 'Amazon MemoryDB: A Fast and Durable Memory-First Cloud Database', 'author': '', 'subject': '-  Information systems  ->  Main memory engines.', 'keywords': '', 'moddate': '2024-04-05T22:57:47+00:00', 'trapped': '', 'modDate': 'D:20240405225747Z', 'creationDate': 'D:20240405225747Z', 'page': 0}, page_content='Amazon MemoryDB: A Fast and Durable Memory-First Cloud\nDatabase\nYacine Taleb\nAmazon Web Services\nCanada\nKevin McGehee

In [113]:
### convert texts to embeddings
texts = [doc.page_content for doc in chunks]
texts

## Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts=texts)

### Store in the vectorDB
vectorStore.add_documents(chunks, embeddings) 

Generate embedding for 88 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00,  5.04it/s]

Generated embeddings with shape: (88, 384)
adding 88 documents to vector store
Successfully added 88 documents to vector Store
Totla document in collection 122


### RAG Retrieval Pipeline From VectorStore

In [114]:
from transformers.models.superpoint.modeling_superpoint import top_k_keypoints


class RAGRetiever:
    def __init__(self,vector_store: VectorStore, embeding_manager: EmbeddingManager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self,query: str, top_k: int = 5, score_threshold : float = 0.0) -> List[Dict[str, Any]]:

        print(f"Retrieving document for query : {query}")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )


            # process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

            
                for i, (doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i+1
                        })
                    
                print(f"Retrieved {len(retrieved_docs)} docs (after filtering)")
            else:
                print("No docs found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise


In [115]:
rag_retriever = RAGRetiever(vectorStore, embedding_manager)
rag_retriever

In [118]:
# rag_retriever.retrieve("What is memoryDB?")
rag_retriever.retrieve("Challenges of Maintaining Durability and Consistency in Redis?")

Retrieving document for query : Challenges of Maintaining Durability and Consistency in Redis?
Top K: 5, Score Threshold: 0.0
Generate embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 docs (after filtering)


[{'id': 'doc_996724ac_11',
  'content': 'element from a set. When executing this command on a primary\nnode, an element in the set is randomly selected. An explicit delete\ncommand for the selected element is then sent to replicas over the\nreplication channel. Following the same pattern, when a Lua script\n(similar to a stored procedure in a traditional RDBMS) is executed,\nthe script itself is not replicated, but the effects it had on the data\nset are recorded and replicated atomically. This mechanism allows\nnon-deterministic operations to be replicated deterministically.\n2.2\nChallenges of Maintaining Durability and\nConsistency in Redis\n2.2.1\nData Loss During Failover Besides serving read-only requests,\nreplica nodes increase availability as they can be promoted to the\nrole of primary within a shard if a primary node fails. Redis uses\na quorum-based approach for both failure detection and election\nof new primaries. Since replicas have a shard’s data available in-\nmemory, 